<a id =tt> </a> <br>
***

# <center> <span style="color:blue;background:yellow"> [WNS Analytics Wizard 2019](https://datahack.analyticsvidhya.com/contest/wns-analytics-wizard-2019/) </span></center>

![](images/wns.png)
*** 

## Notebook Content
1. [Load Libraries](#0)
1. [Load Data](#1)
1. [Preprocessing](#a)
    1. [Process Logs and Items Data](#b)
    1. [Label Encoding](#2)
    1. [Date Columns Processing](#3)
1. [Feature Engineering](#4)
1. [CatBoost](#5)
1. [LightGBM](#l)
1. [Submission DataFrame](#s)

---

<a id=0> </a> <br>

## <span style="background:gray;color:red">1. </span><span style="background:yellow;color:blue">Load Libraries </span>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.metrics import roc_auc_score
import catboost as cgb
import lightgbm as lgb
from bayes_opt import BayesianOptimization

import warnings
warnings.filterwarnings('ignore')

<a id=1> </a> <br>

## <span style="background:gray;color:red">2. </span><span style="background:yellow;color:blue">Load Data </span>

In [ ]:
# Load Data
train = pd.read_csv('Data/train/train.csv')
test = pd.read_csv('Data/test.csv')

logs = pd.read_csv('Data/train/view_log.csv')  # Log details of user ids

items = pd.read_csv('Data/train/item_data.csv') # Item details


### <center> <span style="color:blue;background:yellow"> Data Dictionary </span></center>
    
***

<span style="color:blue;background:gray;font-size:20px">train.csv :</span>

|Variable	|Definition|
|-----------|----------|
|impression_id	|AD impression id|
|impression_time	|Time of the impression at partner website|
|user_id	|user id|
|app_code	|Application Code for a partner website where the ad was shown|
|os_version	|Version of operating system|
|is_4G	|1-Using 4G, 0-No 4G|
|is_click	|(target) Whether user clicked the AD (1-click, 0-no click)|

---
<span style="color:blue;background:gray;font-size:20px">view_log.csv : </span>

|Variable	|Definition|
|-----------|----------|
|server_time	|Timestamp of the log|
|device_type	|Device type of the user|
|session_id	|Browser session id|
|user_id	|user id|
|item_id	|Item id|

---

<span style="color:blue;background:gray;font-size:20px">item_data.csv: </span>

|Variable	|Definition|
|-----------|----------|
|item_id	|Item id|
|item_price	|Price of the item|
|category_1	|Category depth 1|
|category_2	|Category depth 2|
|category_3	|Category depth 3|
|product_type	|anonymized item type|

----
<span style="color:blue;background:gray;font-size:20px"> test.csv: </span> test file contains the impressions for which the participants need to predict the click rate

---
<span style="color:blue;background:gray;font-size:20px">sample_submission.csv: </span> This file contains the format in which you have to submit your predictions.

---

In [ ]:
print('Shape of train Data :', train.shape, '\n Shape of test data :', test.shape)

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train['is_click'].value_counts().plot(kind='bar')

In [ ]:
print('Click percentage :', train['is_click'].value_counts()[1]/train['is_click'].value_counts().sum()*100 , '%')

In [ ]:
items.head()

In [ ]:
items.shape

In [ ]:
for col in items.columns[1:]:
    items[col].plot(kind='box')
    plt.show()


There are no outliers in items' variables except product price.

<a id=a> </a> <br>

## <span style="background:gray;color:red">3. </span><span style="background:yellow;color:blue">Preprocessing </span>

In [ ]:
target = train['is_click'] # Get labels 
train.drop('is_click', axis=1, inplace=True) # Remove labels

<a id=b> </a> <br>

### <span style="background:gray;color:red">3.1 </span><span style="background:yellow;color:blue">Process Logs and Items Data </span>
* Merge Logs and Items DataFrame on `item_id`
* Merge new train and test DataFrames with Logs and Items merge DataFrame on `user_id'
* Create a Pivot tables---> Make  pivot tables with  `impression_id` as index , choose columns and define opeartions on them.
* Reset index of Pivot Tables
* Rename columns 
* Merge Train and Test DataFrame with Pivot Tables on `impression_id` 

In [ ]:
df1 = pd.merge(logs,items, on='item_id', how="left")

In [ ]:
%%time
def process_log_item(df):
    df_new = pd.merge(df,df1, on ="user_id", how="left")
    
    df_new = pd.pivot_table(df_new, index ='impression_id', 
               values=['user_id','item_id', 'item_price','category_1','category_2','category_3', 'product_type'], 
              aggfunc={'user_id':len,'item_id': lambda x:len(set(x)),
                       'item_price':[np.median,np.sum],
                       'category_1':[np.min, np.mean, np.max],
                       'category_2':[np.min, np.mean, np.max],
                       'category_3':[np.min, np.mean, np.max],
                       'product_type':[np.min, np.max]
                      })
    df_new = df_new.reset_index()
    
    df_new.columns = ['impression_id','cat1_max', 'cat1_min', 'cat1_mean',
                    'cat2_max', 'cat2_min', 'cat2_mean',
                    'cat3_max', 'cat3_min', 'cat3_mean',
                     'item_count',
                    'price_median', 'price_sum',
                    'product_type_max', 'product_type_min',
                    'impression_count']
    df = pd.merge(df,df_new, on ="impression_id", how="left" )
    return df

train = process_log_item(train)
test = process_log_item(test)

In [ ]:
train.head()

In [ ]:
rm_col=['impression_id']

In [ ]:
train1 = train.drop(rm_col, axis=1)

test1 = test.drop(rm_col, axis=1)

In [ ]:
print('Shape of train Data :', train1.shape, '\n Shape of test data :', test1.shape)

In [ ]:
train1 = train1.fillna(0)
test1 = test1.fillna(0)

In [ ]:
test1.head()

<a id=2> </a> <br>

### <span style="background:gray;color:red">3.2 </span><span style="background:yellow;color:blue">LabelEncoding </span>

* Do label encoding for `os_version` column

In [ ]:
encode_cols = ['os_version']
def labelencpode(df):
    enc = LabelEncoder()
    for col in encode_cols:
        enc.fit(df[col])
        df[col] = enc.transform(df[col])
    return df
train1 = labelencpode(train1)
test1 = labelencpode(test1)

In [ ]:
train1.head()

<a id=3> </a> <br>

### <span style="background:gray;color:red">3.3 </span><span style="background:yellow;color:blue">Date Columns Processing </span>

* Convert `impression_time` column to datetime format
* Get month, day of week, day of month and hour 

In [ ]:
def date_col(df):
  df['impression_time'] = pd.to_datetime(df['impression_time'])
  df['dow'] = df['impression_time'].apply(lambda x: x.weekday())
  df["day"]  = df['impression_time'].apply(lambda x: x.day)
  df['hour'] = df['impression_time'].apply(lambda x: x.hour)
  df.drop('impression_time', axis=1, inplace=True)
  return df

train1 = date_col(train1)
test1 = date_col(test1)

<a id=4> </a> <br>

## <span style="background:gray;color:red">4. </span><span style="background:yellow;color:blue">Feature Engineering </span>

#### `I found high correlation between categories, item count, product type, and item price. To reduce correlation among them, I took ratios. It did not reduce correlation between target and independent variables.`  

In [ ]:
rm_cols = ['cat1_max', 'cat1_min', 'cat1_mean','cat2_max', 'cat2_min', 'cat2_mean',
                    'cat3_max', 'cat3_min', 'cat3_mean','price_median', 'price_sum',
                    'product_type_max', 'product_type_min', 'item_count', 'impression_count']
def feat_eng(df):
    df['cat1'] = df['cat1_max']/(df['cat1_min'] + 1)
    df['cat12'] = df['cat2_max']/(df['cat2_min'] + 1)
    df['cat3'] = df['cat3_max']/ (df['cat3_min'] + 1)
    df['product_type'] = df['product_type_max'] /(df['product_type_min'] + 1)
    df['price'] = df['price_sum'] / df['price_median']
    df['f1'] = (df['cat1_mean'] + df['cat2_mean'] + df['cat3_mean'])/(df['item_count'] + df['impression_count'] )
    df['f2'] = df['item_count']/df['impression_count']
    df.drop(rm_cols, axis=1, inplace=True)
    return df
train1 = feat_eng(train1)
test1 = feat_eng(test1)

In [ ]:
train1.head()

In [ ]:
print('Shape of train Data :', train1.shape, '\n Shape of test data :', test1.shape)

In [ ]:
train1.head()

In [ ]:
train1['is_click'] = target

In [ ]:
# Plot Correlation 
plt.subplots(figsize=(14,9))
data = train1.corr()
sns.heatmap(data, cmap ='Blues')

In [ ]:
data

<span style='color:blue;background:red;font-size:20px'> `High correlation between independent variables doesn't affect boosting aglorithms as they take one variable at a time for split.Therefore, it is upto you, if you drop highly correlated variable or not.`  </span>

In [ ]:
train1.drop('is_click', axis=1, inplace=True)

In [ ]:
cat_cols = ['user_id','app_code', 'os_version', 'is_4G', 'dow', 'hour', 'day']

In [ ]:
cat_feat =cat_cols

In [ ]:
train1.head()

<a id=5> </a> <br>

## <span style="background:gray;color:red">5. </span><span style="background:yellow;color:blue">CatBoost </span>

In [ ]:
cgb_param = {'learning_rate': 0.05,
             'iterations': 3000,
             'rsm': 0.9474394576478254,
             'l2_leaf_reg': 1.9736243113512275,
             'depth': 5,
             'bagging_temperature': 0.12059739958632439,
             'eval_metric': 'AUC',
             'od_type': 'Iter',
             'od_wait': 100}

In [ ]:
%%time
nfold = 10

skf = StratifiedKFold(n_splits=nfold, shuffle=True, random_state=127)

oof = np.zeros(len(train1))
predictions_cgb = np.zeros(len(test1))

i = 1
for train_index, valid_index in skf.split(target, target.values):
    print("fold {}".format(i))

    train_pool = cgb.Pool(train1.iloc[train_index], target[train_index], cat_features=cat_feat)
    validate_pool = cgb.Pool(train1.iloc[valid_index], target[valid_index], cat_features=cat_feat)
   
    model = cgb.CatBoostClassifier(**cgb_param, use_best_model=True)
    model.fit(train_pool, eval_set=validate_pool, verbose_eval=100)

    oof[valid_index] = model.predict_proba(cgb.Pool(train1.iloc[valid_index], cat_features=cat_feat))[:,1] 

    predictions_cgb += model.predict_proba(cgb.Pool(test1, cat_features=cat_feat))[:,1]/nfold
    i = i + 1

print("\n\nCV AUC-Score: {:<0.5f}".format(roc_auc_score(target, oof)))

In [ ]:
predictions_cgb

<a id=l> </a> <br>

## <span style="background:gray;color:red">6. </span><span style="background:yellow;color:blue">LightGBM </span>

In [ ]:
param_lgb = {'num_leaves': 160,
             'feature_fraction': 0.31712193854996995,
             'bagging_fraction': 0.9765253434754912,
             'max_depth': 17,
             'lambda_l1': 0.8816192486961452,
             'lambda_l2': 2.567616278588324,
             'min_split_gain': 0.08461009792570238,
             'min_child_weight': 1.2653677360323834,
             'learing_rate': 0.01,
             'objective': 'binary',
             'boosting_type': 'gbdt',
             'verbose': 1,
             'metric': {'auc'},
             'is_unbalance': True}

In [ ]:
%%time
nfold = 10

skf = StratifiedKFold(n_splits=nfold, shuffle=True, random_state=127)

oof = np.zeros(len(train1))
predictions_lgb = np.zeros(len(test1))

i = 1
for train_index, valid_index in skf.split(target, target.values):
    print("fold {}".format(i))
    xg_train = lgb.Dataset(train1.iloc[train_index],
                           label=target[train_index],
                           free_raw_data = False
                           )
    xg_valid = lgb.Dataset(train1.iloc[valid_index],
                           label=target[valid_index],
                           free_raw_data = False
                           )   
    
    clf = lgb.train(param_lgb, xg_train, 5000, valid_sets = [xg_valid],
                    categorical_feature=cat_feat,
                    verbose_eval=100, early_stopping_rounds = 100)
    
    oof[valid_index] = clf.predict(train1.iloc[valid_index], num_iteration=clf.best_iteration) 
    
    predictions_lgb += clf.predict(test1, num_iteration=clf.best_iteration) / nfold
    i = i + 1

print("\n\nCV AUC: {:<0.4f}".format(roc_auc_score(target, (oof))))

In [ ]:
predictions_lgb

<a id=s> </a> <br>

## <span style="background:gray;color:red">7</span><span style="background:yellow;color:blue">Submission DataFrame</span>

In [ ]:
predictions = predictions_lgb * 0.05 + predictions_cgb *0.95 

In [ ]:
sub = pd.read_csv('Data/sample_submission.csv')

In [ ]:
sub['is_click'] = predictions

In [ ]:
sub.head()

In [ ]:
sub.to_csv('sub_ensemble.csv', index =False)



## <center><span style="background:gray;color:red">End of the NoteBook</span></center>
***

### <center><span style="background:yellow;color:blue">[GO TO TOP](#tt)</span></center> 